In [1]:
import psycopg2

In [2]:
import pandas as pd

In [3]:
import numpy as np

In [4]:
conn = psycopg2.connect(
    host="localhost",
    database="Movies_11",
    port= "1234",
    user="postgres",
    password="1234")

In [5]:
df = pd.read_sql("""select s.worldwide_box_office, 
s.movie_id, 
AVG(review_score) AS avg_score, 
AVG(review_score)
FILTER (WHERE review_score < 50) AS avg_score_negative, COUNT(*) FILTER (WHERE review_score < 50) AS count_negative,
AVG(review_score)
FILTER (WHERE review_score >= 50) AS avg_score_positive, COUNT(*) FILTER (WHERE review_score >= 50) AS count_positive,
COUNT(*) AS review_count, 
AVG(e.negemo) AS avg_negemo, 
AVG(e.tone) AS avg_tone
from movie_sales s LEFT JOIN expert_review e
ON s.movie_id = e.movie_id
WHERE s.worldwide_box_office is not null AND s.worldwide_box_office > 0 AND e.review_text IS NOT NULL
group by s.movie_id, s.worldwide_box_office
""",conn)

C:\Users\lande\AppData\Local\Temp\ipykernel_22288\478421515.py:1: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql("""select s.worldwide_box_office,


In [6]:
def get_sq3_data(conn): 
    """This returns one film per row with aggregated expert review measures, worldwide box office, grouping of columns like the overall averages and the linguisitic (LIWC) averages
    Films without a worldwide box office and reviews with no text are excluded because LIWC values are placeholders hence the FILTERs. This is a subset of the entire film database (4,823 out of 11,364) due to the exlusion of missing data for the columns mentioned. 
    avg_score_negative is NULL when a film has no review under 50."""
    df = pd.read_sql("""select s.worldwide_box_office, 
    s.movie_id, 
    AVG(review_score) AS avg_score, 
    AVG(review_score)
    FILTER (WHERE review_score < 50) AS avg_score_negative, COUNT(*) FILTER (WHERE review_score < 50) AS count_negative,
    AVG(review_score)
    FILTER (WHERE review_score >= 50) AS avg_score_positive, COUNT(*) FILTER (WHERE review_score >= 50) AS count_positive,
    COUNT(*) AS review_count, 
    AVG(e.negemo) AS avg_negemo, 
    AVG(e.tone) AS avg_tone
    from movie_sales s LEFT JOIN expert_review e
    ON s.movie_id = e.movie_id
    WHERE s.worldwide_box_office is not null AND s.worldwide_box_office > 0 AND e.review_text IS NOT NULL
    group by s.movie_id, s.worldwide_box_office
    """,conn)
    return df


In [7]:
df['count_negative'] >= 5


0        True
1        True
2        True
3        True
4       False
        ...  
4818    False
4819     True
4820     True
4821    False
4822     True
Name: count_negative, Length: 4823, dtype: bool

In [8]:
df_h1 = df[df['count_negative'] >= 5]

In [9]:
df_h1["avg_score_negative"].corr(df_h1["worldwide_box_office"])

np.float64(0.12261852164646993)

In [10]:
df_h1["avg_score_positive"].corr(df_h1["worldwide_box_office"])

np.float64(0.11622174963934895)

In [11]:
df["avg_negemo"].corr(df["worldwide_box_office"])

np.float64(-0.08780483802505659)

In [12]:
df["log_box_office"] = np.log(df["worldwide_box_office"])

In [13]:
df["avg_negemo"].corr(df["log_box_office"])

np.float64(0.010165158562006582)

In [14]:
df["avg_score_negative"].corr(df["log_box_office"])

np.float64(-0.04471643406805525)

In [15]:
df["avg_score_positive"].corr(df["log_box_office"])

np.float64(-0.1085221891189783)